# AnemiaLens: latih ulang segmentasi konjungtiva (versi tahan foto HP)

**Masalah:** U-Net lama hanya belajar dari ±185 foto close-up yang diambil dengan satu alat yang sama. Pada foto HP biasa, terutama **konjungtiva pucat (anemia)**, garisnya sering meleset atau tidak ketemu.

**Perbaikan di notebook ini (hanya model segmentasi; model penilai anemia tidak diubah):**
- **Augmentasi pucat (pallor):** warna konjungtiva digeser ke merah muda keputihan, supaya U-Net mengenali bentuk dan letaknya, bukan sekadar "pita merah".
- **Zoom-out & rotasi/perspektif:** mata dibuat kecil di bingkai, seperti foto HP dari jarak normal.
- **Kondisi kamera HP:** white balance, saturasi, blur, noise, kompresi JPEG, dan resolusi rendah.
- Validasi ganda: Dice pada foto asli **dan** Dice pada simulasi "foto HP".

**Cara pakai:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**. Total sekitar 15–25 menit.
3. Di akhir, file **`seg_unet_v2.pt` (±49 MB)** ter-download. Beri tahu Claude, nanti dipasang ke folder model.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'TIDAK ADA GPU: ganti runtime ke T4 GPU agar cepat'

In [ ]:
!pip install -q timm==1.0.* segmentation-models-pytorch==0.5.* gdown openpyxl

In [ ]:
# Tulis kode AnemiaLens (versi dengan augmentasi robust)
import base64, io, zipfile, sys
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall('/content')
sys.path.insert(0, '/content')
print('Kode siap di /content/anemialens')

## Ambil dataset mask (Eyes-defy-anemia, `archive.zip` ±666 MB)
Hanya dataset ini yang punya mask konjungtiva hasil anotasi manual (217 foto India + Italia).

In [ ]:
import os, zipfile, glob, shutil, subprocess
FID, FNAME = '1jBDBERO18JgmC0nemU5SyUjmrW6BtAYf', 'archive.zip'
dst = f'/content/{FNAME}'
if not os.path.exists(dst):
    subprocess.run(['gdown', f'https://drive.google.com/uc?id={FID}', '-O', dst])
if not os.path.exists(dst) or os.path.getsize(dst) < 1_000_000:
    from google.colab import drive
    drive.mount('/content/drive')          # hanya untuk MENCARI archive.zip, tidak menyimpan apa pun
    found = glob.glob(f'/content/drive/MyDrive/**/{FNAME}', recursive=True)
    if not found:
        raise FileNotFoundError('archive.zip tidak bisa diunduh. Buka link file -> Share -> Anyone with the link.')
    shutil.copy(found[0], dst)
os.makedirs('/content/data', exist_ok=True)
zipfile.ZipFile(dst).extractall('/content/data/eyes_defy_anemia')
for _ in range(3):
    inner = glob.glob('/content/data/**/*.zip', recursive=True)
    if not inner: break
    for z in inner:
        zipfile.ZipFile(z).extractall(os.path.splitext(z)[0]); os.remove(z)
print('OK', os.path.getsize(dst) // 1_000_000, 'MB')

In [ ]:
# Manifest (daftar foto + mask)
import os, sys; sys.path.insert(0, '/content')
from anemialens.common import Config
cfg = Config(work_dir='/content/work', seg_size=384, seg_epochs=60, num_workers=2)
os.makedirs(cfg.work_dir, exist_ok=True); cfg.save(f'{cfg.work_dir}/config.json')
!python /content/anemialens/build_manifest.py /content/data /content/work > /content/manifest.log 2>&1
import pandas as pd
m = pd.read_csv('/content/work/manifest.csv', keep_default_na=False)
print('Foto dengan mask manual:', ((m.mask_path != '') & (m.mask_kind == 'manual')).sum(), '(seharusnya 217)')

## Latih U-Net robust (±10–20 menit di T4)
Perhatikan dua angka per epoch: **Dice val** (foto asli) dan **Dice 'foto HP'** (foto yang sama dibuat seperti foto HP: kecil di bingkai, pucat, blur, JPEG). Model terbaik dipilih dari rata-rata keduanya.

In [ ]:
!python /content/anemialens/train_seg.py /content/work --robust --epochs=60 --no-predict

## Cek visual
Baris atas: foto validasi yang dibuat pucat + zoom-out (hijau = prediksi, kuning = mask asli).
Setelah itu Anda bisa **upload foto sendiri** (mis. dari folder `foto_uji`) untuk melihat garis prediksinya.

In [ ]:
import cv2, numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from anemialens.train_seg import build_unet, predict_mask, augment_robust, pallor
from anemialens.common import read_rgb, read_mask
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = build_unet(False).to(dev); net.load_state_dict(torch.load('/content/work/seg_unet.pt', map_location=dev)); net.eval()
m = pd.read_csv('/content/work/manifest.csv', keep_default_na=False)
m = m[(m.mask_path != '') & (m.mask_kind == 'manual')].sample(6, random_state=1)
rng = np.random.default_rng(7)
fig, ax = plt.subplots(1, 6, figsize=(24, 4))
for a, (_, r) in zip(ax, m.iterrows()):
    im = cv2.resize(read_rgb(r.image_path), (384, 384)); gt = cv2.resize(read_mask(r.mask_path, read_rgb(r.image_path).shape), (384, 384), interpolation=cv2.INTER_NEAREST)
    im = pallor(im, gt, rng, strength=0.7); im, gt = augment_robust(im, gt, rng)
    pr = predict_mask(net, im, 384, dev); o = im.copy()
    cv2.drawContours(o, cv2.findContours(gt.astype(np.uint8), 0, 2)[0], -1, (255, 220, 0), 2)
    cv2.drawContours(o, cv2.findContours(pr.astype(np.uint8), 0, 2)[0], -1, (0, 255, 0), 2)
    a.imshow(o); a.axis('off')
plt.show()
print(open('/content/work/seg_metrics.txt').read())

In [ ]:
# (Opsional) uji dengan foto Anda sendiri
from google.colab import files
up = files.upload()
for name in up:
    im = read_rgb(name); pr = predict_mask(net, im, 384, dev); o = im.copy()
    cv2.drawContours(o, cv2.findContours(pr.astype(np.uint8), 0, 2)[0], -1, (0, 255, 0), max(2, im.shape[1] // 250))
    plt.figure(figsize=(7, 5)); plt.imshow(o); plt.title(f'{name}  | area {pr.mean():.3f}'); plt.axis('off'); plt.show()

## Simpan & download `seg_unet_v2.pt`
Disimpan dalam presisi setengah (fp16) agar ukurannya ±49 MB (hasil prediksi sama). Jika download macet, buka panel 📁 di kiri → klik kanan `seg_unet_v2.pt` → **Download**.

In [ ]:
import torch, shutil
sd = torch.load('/content/work/seg_unet.pt', map_location='cpu')
torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}, '/content/seg_unet_v2.pt')
shutil.copy('/content/work/seg_metrics.txt', '/content/seg_metrics_v2.txt')
print(open('/content/seg_metrics_v2.txt').read())
print('Ukuran:', round(os.path.getsize('/content/seg_unet_v2.pt') / 1e6), 'MB')
from google.colab import files
files.download('/content/seg_unet_v2.pt')
files.download('/content/seg_metrics_v2.txt')